In [1]:
!git clone https://github.com/sabinaibragimova23/smoking-status-prediction.git

%cd /content/smoking-status-prediction/notebooks

import sys
sys.path.append('../src')

fatal: destination path 'smoking-status-prediction' already exists and is not an empty directory.
/content/smoking-status-prediction/notebooks


In [2]:
import os

print("Current directory:", os.getcwd())
print("\nAvailable datasets:")
print(os.listdir("../data/processed"))

Current directory: /content/smoking-status-prediction/notebooks

Available datasets:
['train.csv', 'test_clean.csv', 'test.csv', 'train_clean.csv', 'val_clean.csv', 'val.csv']


In [3]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')
test = pd.read_csv('../data/processed/test.csv')

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

print("\nColumn names:")
print(train.columns.tolist())

print("\nData types:")
print(train.dtypes)

print("\nMissing values:")
print(train.isnull().sum())

print("\nFirst 5 rows:")
display(train.head())

Train shape: (31186, 25)
Validation shape: (6683, 25)
Test shape: (6683, 25)

Column names:
['gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries', 'tartar', 'smoking']

Data types:
gender                  object
age                      int64
height(cm)               int64
weight(kg)               int64
waist(cm)              float64
eyesight(left)         float64
eyesight(right)        float64
hearing(left)          float64
hearing(right)         float64
systolic               float64
relaxation             float64
fasting blood sugar    float64
Cholesterol            float64
triglyceride           float64
HDL                    float64
LDL                    float64
hemoglobin             float64
Urine protein          float64
se

,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,tartar,smoking
0,M,30,175,110,114.0,1.5,0.9,1.0,1.0,140.0,...,122.0,18.1,3.0,1.2,76.0,137.0,73.0,0,Y,1
1,M,35,180,85,91.0,1.2,1.0,1.0,1.0,115.0,...,123.0,15.1,1.0,0.9,17.0,21.0,21.0,0,Y,0
2,M,55,170,70,88.0,1.0,1.2,1.0,1.0,134.0,...,119.0,15.6,1.0,1.0,20.0,16.0,29.0,1,Y,1
3,F,45,150,50,72.0,1.2,1.2,1.0,1.0,88.0,...,124.0,12.7,1.0,0.8,18.0,12.0,17.0,0,Y,0
4,M,55,165,60,78.0,1.5,1.0,1.0,1.0,124.0,...,141.0,16.6,1.0,0.9,23.0,32.0,95.0,0,N,1


In [4]:
# Outlier Detection and Treatment (IQR)

train_clean = train.copy()
val_clean = val.copy()
test_clean = test.copy()

# Continuous numerical features

numerical_cols = [
    'height(cm)', 'weight(kg)', 'waist(cm)',
    'eyesight(left)', 'eyesight(right)',
    'systolic', 'relaxation',
    'fasting blood sugar', 'Cholesterol',
    'triglyceride', 'HDL', 'LDL',
    'hemoglobin', 'serum creatinine',
    'AST', 'ALT', 'Gtp'
]

outlier_report = []

for col in numerical_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)

    IQR = Q3 - Q1

    if IQR == 0:
        continue

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((train[col] < lower) |
             (train[col] > upper)).sum()

    outlier_report.append({
        'Feature': col,
        'Outliers': count,
        'Percentage': round(
            count / len(train) * 100, 2
        )
    })

    for df in [train_clean, val_clean, test_clean]:
        df[col] = df[col].clip(
            lower=lower,
            upper=upper
        )

outlier_df = pd.DataFrame(outlier_report)
outlier_df = outlier_df.sort_values(
    'Outliers', ascending=False
)

display(outlier_df)

print("Train shape:", train_clean.shape)
print("Validation shape:", val_clean.shape)
print("Test shape:", test_clean.shape)

,Feature,Outliers,Percentage
16,Gtp,2707,8.68
15,ALT,2078,6.66
14,AST,2054,6.59
7,fasting blood sugar,1860,5.96
13,serum creatinine,1759,5.64
9,triglyceride,1230,3.94
3,eyesight(left),755,2.42
4,eyesight(right),737,2.36
10,HDL,550,1.76
12,hemoglobin,445,1.43


Train shape: (31186, 25)
Validation shape: (6683, 25)
Test shape: (6683, 25)


In [5]:
# Categorical Encoding
# Check categories before encoding

for col in ['gender', 'tartar']:
    print(f"{col}: {train_clean[col].unique()}")

# Define encoding mappings

gender_mapping = {'M': 1, 'F': 0}
tartar_mapping = {'Y': 1, 'N': 0}

# Apply the same encoding to all datasets

for df in [train_clean, val_clean, test_clean]:
    df['gender'] = df['gender'].map(gender_mapping)
    df['tartar'] = df['tartar'].map(tartar_mapping)

# Validate encoding

for name, df in [
    ('Train', train_clean),
    ('Validation', val_clean),
    ('Test', test_clean)
]:
    assert df[['gender', 'tartar']].notna().all().all(), \
        f"Unexpected category in {name}"

    print(f"{name}: Encoding successful!")

print("\nEncoded dataset preview:")
display(train_clean[['gender', 'tartar', 'smoking']].head())

print("\nData types:")
print(train_clean[['gender', 'tartar']].dtypes)


gender: ['M' 'F']
tartar: ['Y' 'N']
Train: Encoding successful!
Validation: Encoding successful!
Test: Encoding successful!

Encoded dataset preview:


,gender,tartar,smoking
0,1,1,1
1,1,1,0
2,1,1,1
3,0,1,0
4,1,0,1



Data types:
gender    int64
tartar    int64
dtype: object


In [6]:
# Log Transformation
# Potentially right-skewed features

log_candidates = [
    'AST',
    'ALT',
    'Gtp',
    'triglyceride',
    'fasting blood sugar',
    'serum creatinine'
]

# Calculate skewness using training data only

skewness_before = train_clean[log_candidates].skew()

print("Skewness before log transformation:")
print(skewness_before)

# Select strongly right-skewed features

log_features = [
    col for col in log_candidates
    if skewness_before[col] > 0.75
]

print("\nSelected features:", log_features)

# Apply the same transformation to all datasets

for df in [train_clean, val_clean, test_clean]:
    for col in log_features:
        df[col] = np.log1p(df[col].clip(lower=0))

print("\nSkewness after log transformation:")
print(train_clean[log_features].skew())

print("\nDataset shapes:")
print("Train:", train_clean.shape)
print("Validation:", val_clean.shape)
print("Test:", test_clean.shape)


Skewness before log transformation:
AST                    0.803120
ALT                    1.041180
Gtp                    1.133922
triglyceride           0.938557
fasting blood sugar    0.630387
serum creatinine       0.061440
dtype: float64

Selected features: ['AST', 'ALT', 'Gtp', 'triglyceride']

Skewness after log transformation:
AST             0.212142
ALT             0.195827
Gtp             0.304900
triglyceride   -0.029285
dtype: float64

Dataset shapes:
Train: (31186, 25)
Validation: (6683, 25)
Test: (6683, 25)


In [7]:
# Feature Engineering

for df in [train_clean, val_clean, test_clean]:

    # Feature 1: Body Mass Index (BMI)

    height_m = df['height(cm)'] / 100

    df['BMI'] = df['weight(kg)'] / (height_m ** 2)

    # Feature 2: Pulse Pressure

    df['pulse_pressure'] = (
        df['systolic'] - df['relaxation']
    )

print("New features successfully created!")

print("\nTrain shape:", train_clean.shape)
print("Validation shape:", val_clean.shape)
print("Test shape:", test_clean.shape)

print("\nPreview of new features:")
display(
    train_clean[
        ['height(cm)', 'weight(kg)',
         'BMI', 'systolic', 'relaxation',
         'pulse_pressure']
    ].head()
)

print("\nNew feature statistics:")
display(
    train_clean[['BMI', 'pulse_pressure']].describe()
)

New features successfully created!

Train shape: (31186, 27)
Validation shape: (6683, 27)
Test shape: (6683, 27)

Preview of new features:


,height(cm),weight(kg),BMI,systolic,relaxation,pulse_pressure
0,175,105,34.285714,140.0,100.0,40.0
1,180,85,26.234568,115.0,79.0,36.0
2,170,70,24.221453,134.0,72.0,62.0
3,150,50,22.222222,88.0,55.0,33.0
4,165,60,22.038567,124.0,83.0,41.0



New feature statistics:


,BMI,pulse_pressure
count,31186.000000,31186.000000
mean,24.150223,45.476720
std,3.447513,8.735652
min,14.268728,15.000000
25%,21.604938,40.000000
50%,23.875115,45.000000
75%,26.122449,50.000000
max,41.623309,93.000000


In [8]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

# Continuous features to scale

scale_cols = ['age'] + numerical_cols + [
    'BMI', 'pulse_pressure'
]

# Initialize scaler

scaler = StandardScaler()

# Fit ONLY on training data

train_clean[scale_cols] = scaler.fit_transform(
    train_clean[scale_cols]
)

# Transform validation and test using train parameters

val_clean[scale_cols] = scaler.transform(
    val_clean[scale_cols]
)

test_clean[scale_cols] = scaler.transform(
    test_clean[scale_cols]
)

print("Feature Scaling completed successfully!")

print("\nScaled features:")
print(scale_cols)

print("\nTraining data mean and std:")
scaling_report = pd.DataFrame({
    'Mean': train_clean[scale_cols].mean(),
    'Std': train_clean[scale_cols].std(ddof=0)
})

display(scaling_report.round(4))

print("\nFinal dataset shapes:")
print("Train:", train_clean.shape)
print("Validation:", val_clean.shape)
print("Test:", test_clean.shape)

print("\nPreview:")
display(train_clean.head())

Feature Scaling completed successfully!

Scaled features:
['age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'BMI', 'pulse_pressure']

Training data mean and std:


,Mean,Std
age,-0.0,1.0
height(cm),0.0,1.0
weight(kg),0.0,1.0
waist(cm),-0.0,1.0
eyesight(left),0.0,1.0
eyesight(right),0.0,1.0
systolic,0.0,1.0
relaxation,0.0,1.0
fasting blood sugar,0.0,1.0
Cholesterol,-0.0,1.0



Final dataset shapes:
Train: (31186, 27)
Validation: (6683, 27)
Test: (6683, 27)

Preview:


,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,tartar,smoking,BMI,pulse_pressure
0,1,-1.169475,1.129943,3.075720,2.617734,1.554607,-0.299656,1.0,1.0,1.394737,...,3.0,1.717594,1.949112,1.876511,1.565308,0,1,1,2.939990,-0.626949
1,1,-0.757184,1.677344,1.504559,0.980661,0.624575,0.012200,1.0,1.0,-0.483142,...,1.0,0.092692,-1.105470,-0.088798,-0.454248,0,1,0,0.604604,-1.084850
2,1,0.891979,0.582542,0.326188,0.653247,0.004553,0.635911,1.0,1.0,0.944046,...,1.0,0.634326,-0.557399,-0.631139,0.062127,1,1,1,0.020662,1.891507
3,0,0.067398,-1.607062,-1.244974,-1.092964,0.624575,0.635911,1.0,1.0,-2.511252,...,1.0,-0.448942,-0.913238,-1.195429,-0.788344,0,1,0,-0.559253,-1.428276
4,1,0.891979,0.035141,-0.459393,-0.438135,1.554607,0.012200,1.0,1.0,0.192894,...,1.0,0.092692,-0.082638,0.764093,1.805804,0,0,1,-0.612525,-0.512474


In [9]:
# Save Preprocessed Datasets

import os

# Check datasets before saving

for name, df, original in [
    ('Train', train_clean, train),
    ('Validation', val_clean, val),
    ('Test', test_clean, test)
]:
    assert df.shape[0] == original.shape[0]
    assert df.shape[1] == 27
    assert df['smoking'].equals(original['smoking'])
    assert not df.isnull().any().any()
    assert np.isfinite(
        df.select_dtypes(include='number').to_numpy()
    ).all()

    print(f"{name}: All checks passed!")

# Save CSV files

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

train_clean.to_csv(
    f'{output_dir}/train_clean.csv', index=False
)

val_clean.to_csv(
    f'{output_dir}/val_clean.csv', index=False
)

test_clean.to_csv(
    f'{output_dir}/test_clean.csv', index=False
)

print("\nAll cleaned datasets saved successfully!")

# Verify saved files

for filename in [
    'train_clean.csv',
    'val_clean.csv',
    'test_clean.csv'
]:
    path = f'{output_dir}/{filename}'
    df = pd.read_csv(path)

    print(f"{filename}: {df.shape}")

Train: All checks passed!
Validation: All checks passed!
Test: All checks passed!

All cleaned datasets saved successfully!
train_clean.csv: (31186, 27)
val_clean.csv: (6683, 27)
test_clean.csv: (6683, 27)
